> **Niveau 🟡 moyen — les étapes sont données en commentaire, écrivez le code**

# Notebook 3 (bonus) — Pourquoi cette mutation ?

Le gène *HBB* existe en deux versions, ou **allèles** : **A**, l'allèle normal, et **S**, qui porte
la mutation Glu6Val et produit l'hémoglobine S (**HbS**). Chaque personne a deux copies du gène :
AA, AS (un **hétérozygote**, porteur sain) ou SS.

Les personnes **SS** ont la drépanocytose. En Afrique, sans soins, 50 à 90 % d'entre elles meurent
dans les premières années de vie (Grosse et al. 2011), et chaque enfant SS qui meurt emporte deux
copies de l'allèle. Un allèle aussi défavorable devrait devenir rare. Pourtant, dans plusieurs
pays, plus d'une copie du gène *HBB* sur dix porte la mutation.

**Pourquoi ?** Ce notebook cherche la réponse dans les données :
1. où l'allèle est fréquent, et ce qui accompagne sa répartition ;
2. ce que devient une mutation neuve, génération après génération.

**Mode d'emploi**
- `Maj + Entrée` exécute une cellule et passe à la suivante.
- Exécutez les cellules **dans l'ordre**, de haut en bas.
- Les cellules **✔️ Vérification** ne se modifient pas : elles affichent ✅ quand votre code est juste.
- Les graphiques utilisent `matplotlib`, déjà installé dans Colab.

# Partie 1 — Où trouve-t-on l'allèle HbS, et pourquoi là ?

## 0. Charger le tableau

Un pays par ligne. La **fréquence de l'allèle HbS** est la proportion des copies du gène *HBB*
qui portent la mutation, dans la population du pays.

| Colonne | Contenu | Source |
|---|---|---|
| `pays`, `code_iso3` | nom du pays, code ISO à trois lettres | |
| `region_oms` | région de l'Organisation mondiale de la santé (OMS) | OMS |
| `frequence_allele_hbs` | fréquence de l'allèle HbS estimée pour 2010 | Piel et al. 2013 |
| `nombre_enquetes` | nombre d'enquêtes de terrain utilisées pour ce pays (0 : valeur entièrement prédite par le modèle) | Piel et al. 2013 |
| `naissances_ss_par_an` | nombre estimé de nouveau-nés SS par an, en 2010 | Piel et al. 2013 |
| `incidence_paludisme_2000` | cas de paludisme estimés en 2000, pour 1 000 habitants **exposés** ; vide si l'OMS ne publie pas d'estimation (pas de transmission endémique) | OMS |
| `incidence_tuberculose_2000` | cas de tuberculose en 2000, pour 100 000 habitants ; vide : pas de donnée | OMS |
| `prevalence_vih_2000` | pourcentage des adultes de 15 à 49 ans vivant avec le VIH en 2000 (0,1 pour « moins de 0,1 % ») ; vide : pas de donnée | OMS |

Piel et al. ont estimé les fréquences à partir de 1 211 enquêtes publiées, par un modèle
géostatistique : ce sont des estimations, pas des comptages. Les trois dernières colonnes sont
trois maladies infectieuses suivies pays par pays par l'OMS ; 2000 est la première année de la
série du paludisme.

Exécutez la cellule : elle écrit le fichier CSV sur le disque, puis le lit avec `lire_tableau()`.

In [ ]:
#@title ▶️ Exécutez cette cellule pour charger le tableau (ne pas modifier)
# Un pays par ligne. Sources : Piel et al. 2013 (fréquence de l'allèle HbS, naissances SS)
# et OMS, Global Health Observatory (trois maladies) — voir la fin du notebook.

CSV_HBS_PAR_PAYS = """pays,code_iso3,region_oms,frequence_allele_hbs,nombre_enquetes,naissances_ss_par_an,incidence_paludisme_2000,incidence_tuberculose_2000,prevalence_vih_2000
Afghanistan,AFG,Méditerranée orientale,0.0,1,0,84.48,148.0,0.1
Afrique du Sud,ZAF,Afrique,0.003,3,73,3.86,762.0,12.7
Albanie,ALB,Europe,0.014,1,30,,21.0,0.1
Algérie,DZA,Afrique,0.007,18,191,0.02,118.0,0.1
Allemagne,DEU,Europe,0.004,0,85,,12.0,
Angola,AGO,Afrique,0.137,18,8364,326.54,270.0,1.3
Antilles néerlandaises,ANT,Amériques,0.023,2,4,,,
Arabie saoudite,SAU,Méditerranée orientale,0.018,28,691,3.81,23.0,0.1
Argentine,ARG,Amériques,0.0,0,4,2.36,49.0,0.3
Arménie,ARM,Europe,0.0,0,0,0.05,115.0,0.1
Aruba,ABW,Amériques,0.016,1,1,,,
Australie,AUS,Pacifique occidental,0.001,2,5,,6.8,0.1
Autriche,AUT,Europe,0.002,0,1,,22.0,
Azerbaïdjan,AZE,Europe,0.0,0,0,8.12,183.0,0.1
Bahamas,BHS,Amériques,0.024,1,8,,57.0,2.1
Bahreïn,BHR,Méditerranée orientale,0.05,0,59,,57.0,
Bangladesh,BGD,Asie du Sud-Est,0.001,0,16,6.62,221.0,0.1
Barbade,BRB,Amériques,0.025,0,4,,1.1,1.0
Bélarus,BLR,Europe,0.0,0,0,,153.0,
Belgique,BEL,Europe,0.008,18,29,,17.0,0.1
Belize,BLZ,Amériques,0.075,3,47,8.94,113.0,1.2
Bénin,BEN,Afrique,0.159,9,4543,416.0,71.0,1.5
Bhoutan,BTN,Asie du Sud-Est,0.001,0,0,13.44,667.0,0.1
Birmanie,MMR,Asie du Sud-Est,0.0,6,0,34.4,555.0,0.7
Bolivie,BOL,Amériques,0.0,0,0,11.69,402.0,0.4
Bosnie-Herzégovine,BIH,Europe,0.003,0,1,,158.0,
Botswana,BWA,Afrique,0.003,4,4,17.23,976.0,26.0
Brésil,BRA,Amériques,0.017,47,3244,21.54,49.0,0.3
Brunéi Darussalam,BRN,Pacifique occidental,0.0,0,0,,231.0,
Bulgarie,BGR,Europe,0.007,0,17,,83.0,0.1
Burkina Faso,BFA,Afrique,0.056,51,3124,597.17,93.0,2.0
Burundi,BDI,Afrique,0.04,2,889,427.48,213.0,3.3
Cambodge,KHM,Pacifique occidental,0.0,1,0,75.47,853.0,1.2
Cameroun,CMR,Afrique,0.12,34,6915,394.85,84.0,5.0
Canada,CAN,Amériques,0.004,3,45,,8.0,0.2
Cap-Vert,CPV,Afrique,0.026,3,18,1.22,176.0,0.7
Chili,CHL,Amériques,0.0,4,0,,33.0,0.2
Chine,CHN,Pacifique occidental,0.0,8,0,0.02,107.0,
Chypre,CYP,Europe,0.007,1,2,,8.2,
Colombie,COL,Amériques,0.008,2,321,24.35,70.0,0.5
Comores,COM,Afrique,0.018,2,25,66.37,51.0,0.1
Corée du Nord,PRK,Asie du Sud-Est,0.0,0,0,9.79,,
Corée du Sud,KOR,Pacifique occidental,0.0,0,0,1.28,50.0,
Costa Rica,CRI,Amériques,0.016,1,60,1.36,25.0,0.1
Côte d'Ivoire,CIV,Afrique,0.062,7,3567,490.52,166.0,
Croatie,HRV,Europe,0.002,0,1,,56.0,
Cuba,CUB,Amériques,0.03,1,240,,23.0,0.1
Danemark,DNK,Europe,0.0,0,0,,16.0,0.1
Djibouti,DJI,Méditerranée orientale,0.001,0,0,3.07,2700.0,3.4
Égypte,EGY,Méditerranée orientale,0.012,5,1196,0.0,21.0,0.1
Émirats arabes unis,ARE,Méditerranée orientale,0.004,2,4,,8.1,0.1
Équateur,ECU,Amériques,0.0,0,0,14.54,124.0,0.3
Érythrée,ERI,Afrique,0.003,0,9,19.0,1440.0,2.1
Espagne,ESP,Europe,0.007,4,149,,30.0,0.5
Estonie,EST,Europe,0.0,0,0,,105.0,0.1
Eswatini,SWZ,Afrique,0.006,0,4,2.7,981.0,23.1
États-Unis,USA,Amériques,0.011,25,2843,,7.7,
Éthiopie,ETH,Afrique,0.003,8,137,189.18,489.0,3.1
Fidji,FJI,Pacifique occidental,0.0,0,0,,49.0,0.1
Finlande,FIN,Europe,0.0,0,0,,14.0,
France,FRA,Europe,0.013,7,520,,12.0,0.2
Gabon,GAB,Afrique,0.28,3,864,364.66,332.0,4.6
Gambie,GMB,Afrique,0.075,29,418,309.72,187.0,1.6
Géorgie,GEO,Europe,0.0,0,0,5.66,229.0,0.1
Ghana,GHA,Afrique,0.087,52,5474,430.36,216.0,2.6
Grèce,GRC,Europe,0.026,31,206,,9.7,0.1
Grenade,GRD,Amériques,0.037,0,6,,0.0,
Guadeloupe,GLP,Amériques,0.031,1,14,,,
Guam,GUM,Pacifique occidental,0.0,0,0,,,
Guatemala,GTM,Amériques,0.011,0,252,7.21,28.0,
Guinée,GIN,Afrique,0.168,1,5232,415.77,120.0,1.5
Guinée équatoriale,GNQ,Afrique,0.192,2,385,334.3,323.0,
Guinée-Bissau,GNB,Afrique,0.041,4,192,396.46,202.0,3.6
Guyana,GUY,Amériques,0.021,0,17,43.97,148.0,0.8
Guyane française,GUF,Amériques,0.015,6,5,49.83,,
Haïti,HTI,Amériques,0.051,8,1088,9.71,576.0,2.8
Honduras,HND,Amériques,0.049,0,772,8.64,280.0,0.8
Hong Kong,HKG,Pacifique occidental,0.0,0,0,,216.0,
Hongrie,HUN,Europe,0.001,1,0,,45.0,
Îles Salomon,SLB,Pacifique occidental,0.0,0,0,583.82,282.0,
Îles Vierges des États-Unis,VIR,Amériques,0.008,0,0,,,
Inde,IND,Asie du Sud-Est,0.023,113,42016,21.8,322.0,
Indonésie,IDN,Pacifique occidental,0.0,14,1,5.28,370.0,0.1
Irak,IRQ,Méditerranée orientale,0.01,1,629,0.59,60.0,
Iran,IRN,Méditerranée orientale,0.011,0,862,29.07,43.0,
Irlande,IRL,Europe,0.001,0,0,,14.0,0.1
Islande,ISL,Europe,0.0,0,0,,4.6,0.1
Italie,ITA,Europe,0.005,41,91,,9.1,0.2
Jamaïque,JAM,Amériques,0.037,4,137,,12.0,1.7
Japon,JPN,Pacifique occidental,0.0,1,0,,47.0,
Jordanie,JOR,Méditerranée orientale,0.017,2,145,,9.5,
Kazakhstan,KAZ,Europe,0.0,0,0,,168.0,0.1
Kenya,KEN,Afrique,0.038,51,4475,220.26,451.0,9.2
Kirghizistan,KGZ,Europe,0.0,0,0,0.0,291.0,0.1
Koweït,KWT,Méditerranée orientale,0.031,0,116,,36.0,0.1
La Réunion,REU,Afrique,0.001,0,0,,,
Laos,LAO,Pacifique occidental,0.0,0,0,31.76,330.0,0.1
Lesotho,LSO,Afrique,0.001,0,0,,827.0,22.0
Lettonie,LVA,Europe,0.0,0,0,,196.0,0.3
Liban,LBN,Méditerranée orientale,0.049,1,251,,28.0,0.1
Libéria,LBR,Afrique,0.046,15,562,397.08,115.0,1.4
Libye,LBY,Méditerranée orientale,0.021,4,188,,59.0,0.1
Lituanie,LTU,Europe,0.0,0,0,,181.0,0.1
Luxembourg,LUX,Europe,0.005,0,0,,14.0,0.1
Madagascar,MDG,Afrique,0.061,37,3379,54.85,207.0,0.1
Malaisie,MYS,Pacifique occidental,0.0,6,0,13.83,152.0,0.4
Malawi,MWI,Afrique,0.033,15,1688,452.55,386.0,13.9
Maldives,MDV,Asie du Sud-Est,0.0,0,0,,139.0,0.1
Mali,MLI,Afrique,0.057,8,2701,394.98,76.0,1.7
Malte,MLT,Europe,0.015,0,2,,4.0,0.1
Maroc,MAR,Méditerranée orientale,0.003,0,40,0.0,114.0,0.1
Martinique,MTQ,Amériques,0.035,1,12,,,
Maurice,MUS,Afrique,0.0,0,0,,32.0,0.3
Mauritanie,MRT,Afrique,0.05,1,433,96.39,229.0,0.3
Mayotte,MYT,Afrique,0.027,0,12,,,
Mexique,MEX,Amériques,0.007,34,641,3.53,37.0,0.2
Micronésie,FSM,Pacifique occidental,0.0,0,0,,281.0,
Moldavie,MDA,Europe,0.0,0,0,,70.0,0.5
Mongolie,MNG,Pacifique occidental,0.0,0,0,,428.0,0.1
Monténégro,MNE,Europe,0.005,0,1,,,0.1
Mozambique,MOZ,Afrique,0.027,14,1645,473.67,336.0,8.8
Namibie,NAM,Afrique,0.01,8,30,53.57,985.0,13.5
Népal,NPL,Asie du Sud-Est,0.002,1,11,6.73,417.0,0.2
Nicaragua,NIC,Amériques,0.023,0,204,6.78,125.0,
Niger,NER,Afrique,0.08,7,4965,359.24,83.0,0.8
Nigeria,NGA,Afrique,0.171,24,85186,401.79,219.0,2.0
Norvège,NOR,Europe,0.0,0,0,,7.0,0.1
Nouvelle-Calédonie,NCL,Pacifique occidental,0.0,0,0,,,
Nouvelle-Zélande,NZL,Pacifique occidental,0.0,0,0,,12.0,0.1
Oman,OMN,Méditerranée orientale,0.001,2,0,0.0,25.0,0.1
Ouganda,UGA,Afrique,0.082,11,10143,482.77,276.0,8.5
Ouzbékistan,UZB,Europe,0.0,0,0,5.08,156.0,0.2
Pakistan,PAK,Méditerranée orientale,0.002,1,228,6.08,275.0,0.1
Panama,PAN,Amériques,0.03,3,140,0.54,71.0,0.7
Papouasie-Nouvelle-Guinée,PNG,Pacifique occidental,0.0,1,0,264.08,1060.0,0.7
Paraguay,PRY,Amériques,0.003,0,7,37.32,98.0,0.1
Pays-Bas,NLD,Europe,0.011,2,82,,8.9,0.1
Pérou,PER,Amériques,0.0,10,0,13.19,367.0,0.4
Philippines,PHL,Pacifique occidental,0.0,0,0,1.73,590.0,0.1
Pologne,POL,Europe,0.0,0,0,,46.0,
Polynésie française,PYF,Pacifique occidental,0.0,0,0,,,
Porto Rico,PRI,Amériques,0.004,1,2,,9.8,
Portugal,PRT,Europe,0.01,2,41,,59.0,0.5
Qatar,QAT,Méditerranée orientale,0.032,0,41,,99.0,0.1
République centrafricaine,CAF,Afrique,0.077,2,976,438.12,681.0,7.9
République démocratique du Congo,COD,Afrique,0.165,11,38217,458.24,612.0,2.1
République dominicaine,DOM,Amériques,0.019,0,247,0.32,145.0,1.4
République du Congo,COG,Afrique,0.145,4,1560,349.6,569.0,4.5
Roumanie,ROU,Europe,0.001,0,2,,189.0,0.1
Royaume-Uni,GBR,Europe,0.009,4,296,,12.0,
Russie,RUS,Europe,0.0,0,0,,95.0,
Rwanda,RWA,Afrique,0.023,2,601,180.43,238.0,4.8
Saint-Vincent-et-les-Grenadines,VCT,Amériques,0.045,0,6,,14.0,
Sainte-Lucie,LCA,Amériques,0.049,1,12,,5.7,0.2
Salvador,SLV,Amériques,0.014,1,80,0.62,58.0,0.5
Samoa,WSM,Pacifique occidental,0.0,0,0,,24.0,
Sao Tomé-et-Principe,STP,Afrique,0.094,0,41,222.02,254.0,1.9
Sénégal,SEN,Afrique,0.067,20,2535,230.37,173.0,0.7
Serbie,SRB,Europe,0.004,0,10,,,0.1
Sierra Leone,SLE,Afrique,0.164,0,2838,424.58,604.0,1.7
Singapour,SGP,Pacifique occidental,0.0,1,0,,77.0,0.1
Slovaquie,SVK,Europe,0.001,0,0,,32.0,
Slovénie,SVN,Europe,0.001,0,0,,27.0,0.1
Somalie,SOM,Méditerranée orientale,0.002,2,13,133.15,352.0,0.3
Soudan,SDN,Méditerranée orientale,0.043,32,4567,89.72,148.0,0.1
Sri Lanka,LKA,Asie du Sud-Est,0.001,28,2,47.33,148.0,0.1
Suède,SWE,Europe,0.0,0,0,,6.7,0.1
Suisse,CHE,Europe,0.004,0,6,,11.0,0.2
Surinam,SUR,Amériques,0.03,0,19,161.38,50.0,1.1
Syrie,SYR,Méditerranée orientale,0.025,4,901,0.0,79.0,0.1
Tadjikistan,TJK,Europe,0.0,0,0,9.08,140.0,0.1
Tanzanie,TZA,Afrique,0.074,57,11022,334.61,503.0,6.1
Tchad,TCD,Afrique,0.051,6,2045,272.13,123.0,2.5
Tchéquie,CZE,Europe,0.001,0,0,,18.0,
Thaïlande,THA,Asie du Sud-Est,0.0,0,0,6.57,241.0,2.9
Timor oriental,TLS,Asie du Sud-Est,0.0,0,0,158.38,,0.1
Togo,TGO,Afrique,0.125,4,2175,445.12,51.0,3.9
Tonga,TON,Pacifique occidental,0.0,0,0,,23.0,
Trinité-et-Tobago,TTO,Amériques,0.038,1,56,,34.0,1.4
Tunisie,TUN,Méditerranée orientale,0.011,12,81,,57.0,0.1
Turkménistan,TKM,Europe,0.0,0,0,0.08,193.0,
Turquie,TUR,Europe,0.005,45,330,2.69,54.0,
Ukraine,UKR,Europe,0.0,0,0,,193.0,
Uruguay,URY,Amériques,0.001,0,1,,33.0,0.3
Vanuatu,VUT,Pacifique occidental,0.0,0,0,124.37,331.0,
Vénézuela,VEN,Amériques,0.014,40,435,2.9,62.0,0.3
Viêt Nam,VNM,Pacifique occidental,0.0,0,0,3.54,296.0,0.2
Yémen,YEM,Méditerranée orientale,0.004,1,80,83.3,94.0,0.1
Zambie,ZMB,Afrique,0.112,11,5652,360.33,759.0,15.0
Zimbabwe,ZWE,Afrique,0.021,7,483,102.59,605.0,26.0
"""

import csv

with open("hbs_par_pays.csv", "w", encoding="utf-8") as f:
    f.write(CSV_HBS_PAR_PAYS)

COLONNES_MALADIES = ["incidence_paludisme_2000", "incidence_tuberculose_2000", "prevalence_vih_2000"]


def lire_tableau(nom_fichier):
    """Lit un fichier CSV : chaque ligne devient un dictionnaire {nom de colonne: valeur}.
    Convertit les colonnes numériques ; une case vide devient None."""
    tableau = []
    with open(nom_fichier, encoding="utf-8") as f:
        for ligne in csv.DictReader(f):
            ligne["frequence_allele_hbs"] = float(ligne["frequence_allele_hbs"])
            ligne["nombre_enquetes"] = int(ligne["nombre_enquetes"])
            ligne["naissances_ss_par_an"] = int(ligne["naissances_ss_par_an"])
            for colonne in COLONNES_MALADIES:
                if ligne[colonne] == "":
                    ligne[colonne] = None
                else:
                    ligne[colonne] = float(ligne[colonne])
            tableau.append(ligne)
    return tableau


tableau = lire_tableau("hbs_par_pays.csv")
print(len(tableau), "pays chargés ✅")

`lire_tableau()` renvoie une **liste de dictionnaires** : un dictionnaire par pays, dont les clés
sont les noms de colonnes. `tableau[0]["pays"]` est le nom du premier pays. Une case vide du
fichier devient `None`.

In [ ]:
# le tableau est une liste ; chaque élément est un dictionnaire (une ligne du fichier)
print("nombre de lignes :", len(tableau))
print("colonnes :", list(tableau[0]))
print()
for ligne in tableau[:3]:
    print(ligne)

## 1. Où l'allèle est-il le plus fréquent ? Où naissent le plus d'enfants SS ?

Classez les pays deux fois : par fréquence de l'allèle, puis par nombre de naissances SS par an.

`sorted(liste, key=..., reverse=True)` renvoie une **nouvelle** liste triée, du plus grand au plus
petit ; `key` est une fonction qui, pour chaque élément, donne la valeur à comparer. Ici, chaque
élément est un dictionnaire : `key=lambda ligne: ligne["pays"]` trierait par nom.

In [ ]:
# 1. par_frequence = le tableau trié par fréquence de l'allèle, de la plus grande à la
#    plus petite : sorted(tableau, key=lambda ligne: ..., reverse=True)
# 2. afficher les 10 premiers : pays et fréquence (tranche [:10], f-string)
# 3. par_naissances_ss = le tableau trié par nombre de naissances SS par an, décroissant
# 4. afficher les 10 premiers : pays, naissances SS par an, fréquence de l'allèle
pass  # ← remplacez cette ligne par votre code

In [ ]:
# ✔️ Vérification — exécutez sans modifier
if len(par_frequence) == 190 and len(par_naissances_ss) == 190:
    print("✅ les deux classements contiennent les 190 pays")
else:
    print("❌ un classement doit contenir tout le tableau (190 pays) : utilisez sorted(tableau, ...)")

if par_frequence[0]["pays"] == "Gabon":
    print("✅ allèle le plus fréquent : Gabon (0,280)")
else:
    print("❌ attendu Gabon en tête du classement par fréquence, trouvé", par_frequence[0]["pays"])

if par_naissances_ss[0]["pays"] == "Nigeria" and par_naissances_ss[1]["pays"] == "Inde":
    print("✅ le plus de naissances SS : Nigeria (85 186 par an), puis Inde (42 016 par an),"
          " où l'allèle n'est qu'à 0,023")
else:
    print("❌ attendu Nigeria puis Inde en tête du classement par naissances SS, trouvé",
          par_naissances_ss[0]["pays"], "et", par_naissances_ss[1]["pays"])

Les deux classements ne se ressemblent pas : le nombre d'enfants SS dépend de la fréquence de
l'allèle **et** du nombre de naissances dans le pays.

## 2. Par région

Pour chaque région OMS : le nombre de pays, la fréquence moyenne de l'allèle (moyenne simple des
pays, sans tenir compte de leur population), et la part des naissances SS mondiales.

On accumule dans des **dictionnaires** dont la clé est la région : on parcourt le tableau une
fois, et chaque ligne ajoute sa valeur au total de sa région.

In [ ]:
# 1. trois dictionnaires vides : nombre_pays, somme_frequences, naissances_ss
#    (clé : la région OMS)
# 2. pour chaque ligne du tableau :
#      si sa région n'est pas encore une clé, la créer avec 0 dans les trois dictionnaires
#      ajouter 1 au nombre de pays, la fréquence de l'allèle à la somme des fréquences,
#      les naissances SS par an aux naissances SS
# 3. total_ss = somme de toutes les naissances SS (sum(naissances_ss.values()))
# 4. frequence_moyenne = dictionnaire région → somme des fréquences / nombre de pays
# 5. pour chaque région : afficher le nombre de pays, la fréquence moyenne, et la part
#    des naissances SS mondiales en % (100 * naissances_ss[region] / total_ss)
pass  # ← remplacez cette ligne par votre code

In [ ]:
# ✔️ Vérification — exécutez sans modifier
FREQUENCE_MOYENNE_ATTENDUE = {'Méditerranée orientale': 0.01652381, 'Afrique': 0.069957447, 'Europe': 0.003583333, 'Amériques': 0.021051282, 'Pacifique occidental': 4e-05, 'Asie du Sud-Est': 0.0028}
if len(frequence_moyenne) == 6 and all(
        abs(frequence_moyenne[region] - valeur) < 1e-6
        for region, valeur in FREQUENCE_MOYENNE_ATTENDUE.items()):
    print("✅ fréquence moyenne la plus haute : Afrique (0,070)")
else:
    print("❌ fréquences moyennes attendues :", FREQUENCE_MOYENNE_ATTENDUE)

if total_ss == 288811 and naissances_ss["Afrique"] == 223792:
    print("✅ région Afrique : 77,5 % des naissances SS ; Asie du Sud-Est : 14,6 %,"
          " presque toutes en Inde")
else:
    print("❌ attendu", 288811, "naissances SS au total, dont", 223792,
          "en Afrique ; trouvé", total_ss, "et", naissances_ss.get("Afrique"))

Remarque : la somme des naissances SS de tous les pays (288 811) est plus
faible que le total mondial publié par Piel et al. (312 302). Chaque valeur du tableau est la
médiane d'une distribution de valeurs possibles, et la somme des médianes n'est pas la médiane de
la somme.

**Question.** Chaque génération, les enfants SS qui meurent emportent deux copies de l'allèle.
Que faut-il pour que l'allèle reste fréquent malgré tout, génération après génération ? Et
pourquoi seulement dans certaines régions ?

✍️ *Votre réponse (double-cliquez sur cette cellule pour écrire) :*

## 3. Trois maladies candidates

Le tableau contient trois maladies infectieuses suivies par l'OMS : la tuberculose, le VIH et le
paludisme. Si l'une d'elles maintient l'allèle, sa répartition doit ressembler à celle de
l'allèle : les pays où elle est fréquente doivent être ceux où l'allèle est fréquent.

Un **nuage de points** par maladie : un point par pays, la valeur de la maladie en abscisse, la
fréquence de l'allèle en ordonnée, une couleur par région.

- `fig, axes = plt.subplots(1, 3, ...)` crée trois graphiques côte à côte ; `axes` est la liste
  des trois.
- `zip(MALADIES, axes)` parcourt ensemble les maladies et les graphiques.
- `axe.scatter(x, y, ...)` trace un point pour chaque paire `(x[i], y[i])` ; en l'appelant une
  fois par région, avec une couleur et un `label` différents, `legend()` affiche la légende.

In [ ]:
import matplotlib.pyplot as plt

# une couleur par région OMS
COULEURS = {
    "Afrique": "tab:red",
    "Méditerranée orientale": "tab:orange",
    "Asie du Sud-Est": "tab:purple",
    "Europe": "tab:green",
    "Amériques": "tab:blue",
    "Pacifique occidental": "tab:brown",
}
# les trois maladies : colonne du tableau → titre de l'axe
MALADIES = {
    "incidence_tuberculose_2000": "tuberculose : cas pour 100 000 habitants (2000)",
    "prevalence_vih_2000": "VIH : % des adultes de 15 à 49 ans (2000)",
    "incidence_paludisme_2000": "paludisme : cas pour 1 000 habitants exposés (2000)",
}
# les pays dont le nom sera écrit à côté du point (codes ISO3)
PAYS_ANNOTES = ["GAB", "NGA", "IND", "ETH", "BWA"]

In [ ]:
# 1. fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True) : 3 graphiques
# 2. pour chaque (colonne, axe) de zip(MALADIES, axes) :
#      pour chaque région de COULEURS :
#        lignes = les pays de cette région dont la valeur de `colonne` n'est pas None
#        axe.scatter(valeurs de la maladie, fréquences de l'allèle,
#                    color=COULEURS[region], label=region, alpha=0.7, s=18)
#      écrire le nom des pays de PAYS_ANNOTES à côté de leur point (axe.annotate)
#      titre de l'axe horizontal : axe.set_xlabel(MALADIES[colonne])
# 3. axes[0].set_ylabel(...), axes[0].legend(fontsize=8), plt.show()
pass  # ← remplacez cette ligne par votre code

Pour comparer les trois maladies avec un nombre plutôt qu'à l'œil : le **coefficient de
corrélation de Pearson** `r`. Il mesure, entre −1 et 1, à quel point deux grandeurs varient
ensemble de façon linéaire : 1 si les points sont alignés sur une droite montante, 0 s'il n'y a
pas de relation linéaire, −1 sur une droite descendante.

$$r = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_i (x_i - \bar{x})^2 \; \sum_i (y_i - \bar{y})^2}}$$

où $\bar{x}$ et $\bar{y}$ sont les moyennes. Pour chaque maladie, on le calcule sur les pays qui
ont une valeur ; sur tous les pays, puis sur les seuls pays de la région Afrique.

In [ ]:
def correlation(x, y):
    """Coefficient de corrélation de Pearson entre deux listes de même longueur."""
    # 1. mx, my : les moyennes de x et de y
    # 2. somme_produits = somme des (a - mx) * (b - my), pour a, b dans zip(x, y)
    # 3. somme_carres_x = somme des (a - mx) ** 2 ; de même somme_carres_y
    # 4. renvoyer somme_produits / (somme_carres_x * somme_carres_y) ** 0.5
    pass  # ← remplacez cette ligne par votre code


def correlation_avec_allele(lignes, colonne):
    """Nombre de lignes qui ont une valeur dans `colonne`, et r entre cette colonne et
    la fréquence de l'allèle HbS, sur ces lignes."""
    # 5. garder les lignes où ligne[colonne] n'est pas None
    # 6. renvoyer leur nombre, et correlation(valeurs de la colonne, fréquences)


# 7. afrique = les lignes de la région "Afrique"
# 8. r_tous et r_afrique : deux dictionnaires colonne → r, remplis pour chaque colonne
#    de MALADIES, sur tout le tableau et sur afrique ; afficher les résultats

In [ ]:
# ✔️ Vérification — exécutez sans modifier
essais = (correlation([1, 2, 3], [2, 4, 6]), correlation([1, 2, 3], [3, 2, 1]))
if abs(essais[0] - 1) < 1e-9 and abs(essais[1] + 1) < 1e-9:
    print("✅ correlation() donne 1 pour deux listes qui croissent ensemble, -1 pour deux"
          " listes qui varient en sens contraire")
else:
    print("❌ correlation([1, 2, 3], [2, 4, 6]) doit valoir 1, et correlation([1, 2, 3], [3, 2, 1])"
          " -1 ; obtenu", essais)

R_TOUS_ATTENDU = {'incidence_tuberculose_2000': 0.07510480904698158, 'prevalence_vih_2000': 0.1298943140552818, 'incidence_paludisme_2000': 0.6730059691185322}
R_AFRIQUE_ATTENDU = {'incidence_tuberculose_2000': -0.23692879823060997, 'prevalence_vih_2000': -0.3298373307269342, 'incidence_paludisme_2000': 0.5354394456109991}
if (all(abs(r_tous[c] - v) < 1e-6 for c, v in R_TOUS_ATTENDU.items())
        and all(abs(r_afrique[c] - v) < 1e-6 for c, v in R_AFRIQUE_ATTENDU.items())):
    print("✅ tous les pays : tuberculose r = +0,08, VIH r = +0,13, paludisme r = +0,67")
    print("✅ région Afrique : tuberculose r = −0,24, VIH r = −0,33, paludisme r = +0,54")
else:
    print("❌ valeurs attendues, tous les pays :", R_TOUS_ATTENDU)
    print("❌ région Afrique :", R_AFRIQUE_ATTENDU)

**Question.** Laquelle des trois maladies suit la répartition de l'allèle ? Pourquoi refaire le
calcul sur les seuls pays de la région Afrique ? Une corrélation entre pays suffit-elle à
conclure ?

✍️ *Votre réponse (double-cliquez sur cette cellule pour écrire) :*

## 4. Les porteurs AS sont-ils moins touchés ?

Une **étude cas-témoins** compare la fréquence d'une caractéristique chez des malades (les cas)
et chez des personnes comparables qui n'ont pas cette maladie (les témoins). En Gambie, Hill et
al. (1991) ont déterminé le génotype d'enfants de moins de 10 ans, répartis en quatre groupes
(chiffres repris par Taylor et al. 2012) :

| Groupe | Enfants | Enfants AS |
|---|---|---|
| paludisme grave | 619 | 1,2 % |
| paludisme simple | 354 | 2,8 % |
| maladie grave, sans paludisme | 332 | 10,9 % |
| maladie bénigne, sans paludisme | 510 | 12,9 % |

Pour comparer un groupe au groupe de référence (maladie bénigne, sans paludisme), on calcule le
**rapport des cotes** (*odds ratio*). La **cote** d'être AS dans un groupe vaut p / (1 − p), où p
est la proportion d'enfants AS ; le rapport des cotes est la cote du groupe divisée par celle du
groupe de référence. Un rapport de 1 : pas de différence ; très inférieur à 1 : les AS sont
beaucoup plus rares dans ce groupe.

In [ ]:
# Gambie, enfants de moins de 10 ans (Hill et al. 1991, repris par Taylor et al. 2012) :
# groupe → (nombre d'enfants, pourcentage d'enfants AS)
GROUPES = {
    "paludisme grave": (619, 1.2),
    "paludisme simple": (354, 2.8),
    "maladie grave, sans paludisme": (332, 10.9),
    "maladie bénigne, sans paludisme": (510, 12.9),
}

In [ ]:
def cote(pourcentage):
    """Cote d'être AS : p / (1 - p), avec p = pourcentage / 100."""
    pass  # ← remplacez cette ligne par votre code


# 1. pourcentage_reference = le pourcentage d'AS du groupe "maladie bénigne, sans paludisme"
#    (GROUPES[...] est un couple (effectif, pourcentage) : [1] pour le pourcentage)
# 2. rapports = dictionnaire groupe → cote(pourcentage du groupe) / cote(pourcentage_reference)
#    (for groupe, (effectif, pourcentage_as) in GROUPES.items())
# 3. afficher, pour chaque groupe : effectif, pourcentage d'AS, rapport des cotes

In [ ]:
# ✔️ Vérification — exécutez sans modifier
if abs(cote(50) - 1) < 1e-9 and abs(cote(20) - 0.25) < 1e-9:
    print("✅ cote(50) = 1 (une chance sur deux) ; cote(20) = 0,25 (1 contre 4)")
else:
    print("❌ cote(50) doit valoir 1 et cote(20) 0,25 ; obtenu", cote(50), "et", cote(20))

RAPPORTS_ATTENDUS = {'paludisme grave': 0.08200734394124846, 'paludisme simple': 0.19450027115832452, 'maladie grave, sans paludisme': 0.8259946580360017, 'maladie bénigne, sans paludisme': 1.0}
if len(rapports) == 4 and all(abs(rapports[g] - v) < 1e-6 for g, v in RAPPORTS_ATTENDUS.items()):
    print("✅ paludisme grave : 0,08 ; paludisme simple : 0,19 ; maladie grave, sans paludisme : 0,83 ; maladie bénigne, sans paludisme : 1,00")
else:
    print("❌ rapports des cotes attendus :", RAPPORTS_ATTENDUS)

**Question.** Que concluez-vous ? À quoi sert le groupe « maladie grave, sans paludisme » ?

✍️ *Votre réponse (double-cliquez sur cette cellule pour écrire) :*

**L'hypothèse du paludisme.** Les deux résultats vont dans le même sens. Une méta-analyse de cinq
études cas-témoins (plus de 10 000 patients) donne, pour le paludisme grave, un rapport des cotes
de **0,09** (IC 95 % : 0,06–0,12) entre enfants AS et AA (Taylor et al. 2012). C'est l'hypothèse
formulée par J. B. S. Haldane en 1949 pour les thalassémies, et par A. C. Allison en 1954 pour
HbS : là où le paludisme est intense, les AS survivent mieux que les AA, tandis que les SS meurent
jeunes. Deux effets opposés, qui maintiennent l'allèle.

## 5. Les pays qui s'écartent de la tendance

La cellule suivante liste deux groupes de pays : ceux où l'allèle est présent sans estimation de
paludisme en 2000, et ceux où le paludisme est fréquent et l'allèle rare.

In [ ]:
par_frequence_decroissante = sorted(tableau, key=lambda ligne: ligne["frequence_allele_hbs"],
                                    reverse=True)

print("Allèle HbS ≥ 0,02, sans estimation OMS du paludisme en 2000 :")
for ligne in par_frequence_decroissante:
    if ligne["frequence_allele_hbs"] >= 0.02 and ligne["incidence_paludisme_2000"] is None:
        print(f"  {ligne['pays']:<34} {ligne['frequence_allele_hbs']:.3f}   {ligne['region_oms']}")

print()
print("Paludisme > 150 cas pour 1 000 en 2000, allèle HbS < 0,05 :")
for ligne in par_frequence_decroissante:
    incidence = ligne["incidence_paludisme_2000"]
    if incidence is not None and incidence > 150 and ligne["frequence_allele_hbs"] < 0.05:
        print(f"  {ligne['pays']:<34} {ligne['frequence_allele_hbs']:.3f}   {incidence:6.1f}   {ligne['region_oms']}")

**Question.**
1. Premier groupe : l'allèle HbS est présent, et l'OMS n'estime aucune transmission du paludisme.
   Distinguez deux situations, d'après les régions.
2. Second groupe : le paludisme est fréquent et l'allèle rare. Quelles explications proposer ?
3. Pourquoi l'incidence du paludisme en 2000 n'est-elle pas la bonne mesure pour expliquer la
   fréquence d'un allèle ?

✍️ *Votre réponse (double-cliquez sur cette cellule pour écrire) :*

# Partie 2 — Une mutation neuve : se perd-elle ou s'installe-t-elle ?

La partie 1 a mis en évidence deux effets opposés : les AS sont protégés contre le paludisme, les
SS sont malades. Reste à comprendre comment une mutation neuve, portée au départ par une seule
personne, devient fréquente.

L'analyse des génomes de 156 porteurs de l'allèle HbS indique que la mutation est apparue **une
seule fois**, il y a environ **259 générations** (IC 95 % : 123–395), soit environ **7 300 ans**
avec des générations de 28 ans (Shriner et Rotimi 2018).

À son apparition, la mutation est portée par **une seule copie** du gène, parmi les 2N copies
d'une population de N personnes. Deux forces décident de son sort :

- **La sélection.** La **valeur sélective** d'un génotype mesure sa contribution à la génération
  suivante, relative à celle d'un génotype de référence ; ici, sa survie jusqu'à l'âge adulte.
  Chez 12 387 adultes yoruba d'Ibadan (Nigeria), AA 9 365, AS 2 993, SS 29 : comparés aux
  effectifs attendus si le génotype ne changeait pas la survie, on obtient w_AA = 0,88, w_AS = 1
  et w_SS = 0,14 (données reprises par F. J. Ayala, *Encyclopædia Britannica*, article
  « Evolution »). À nombre de naissances égal, pour 100 AS qui atteignent l'âge adulte, 88 AA et
  14 SS l'atteignent.
- **La dérive génétique.** Les 2N copies de la génération suivante sont tirées au hasard parmi
  les copies des parents. Une copie rare peut ne pas être transmise du tout, par hasard.

Le modèle de **Wright-Fisher** simule ces deux forces, une génération à la fois :

1. fréquence de HbS : `q = copies / (2N)`, et `p = 1 − q` ;
2. fréquence de HbS après sélection :

$$q_{sel} = \frac{p\,q\,w_{AS} + q^2\,w_{SS}}{p^2\,w_{AA} + 2\,p\,q\,w_{AS} + q^2\,w_{SS}}$$

3. génération suivante : on tire 2N copies, chacune est HbS avec la probabilité `q_sel`.

In [ ]:
import random

# valeurs sélectives relatives en zone d'endémie du paludisme
# (12 387 adultes yoruba d'Ibadan, Nigeria ; l'hétérozygote AS sert de référence : 1)
W_AA = 0.88
W_AS = 1
W_SS = 0.14

# taille de la population simulée (nombre de personnes ; 2 × N copies du gène HBB)
N = 1000

## 6. Une génération

Écrivez `une_generation(copies, N, w_AA, w_AS, w_SS)`, qui renvoie le nombre de copies de HbS à la
génération suivante.

`random.random()` renvoie un nombre au hasard entre 0 et 1 : `random.random() < q_sel` est vrai
avec la probabilité `q_sel`.

In [ ]:
def une_generation(copies, N, w_AA, w_AS, w_SS):
    """Nombre de copies de HbS à la génération suivante, pour `copies` copies parmi 2N."""
    # 1. q = copies / (2 * N), p = 1 - q
    # 2. valeur_selective_moyenne = p² w_AA + 2pq w_AS + q² w_SS
    #    q_sel = (pq w_AS + q² w_SS) / valeur_selective_moyenne
    # 3. compter, sur 2N tirages, combien de fois random.random() < q_sel
    # 4. renvoyer ce nombre
    pass  # ← remplacez cette ligne par votre code


# 5. essai : afficher 20 fois une_generation(1, N, W_AA, W_AS, W_SS)

In [ ]:
# ✔️ Vérification — exécutez sans modifier
random.seed(0)
tirages = [une_generation(1, 1000, 0.88, 1, 0.14) for _ in range(2000)]
if all(isinstance(t, int) and 0 <= t <= 2000 for t in tirages):
    print("✅ une_generation() renvoie un nombre entier de copies, entre 0 et 2N")
else:
    print("❌ une_generation() doit renvoyer un entier entre 0 et 2N ; exemples obtenus :", tirages[:5])

if une_generation(0, 1000, 0.88, 1, 0.14) == 0 and une_generation(2000, 1000, 1, 1, 1) == 2000:
    print("✅ un allèle absent reste absent ; un allèle présent sur toutes les copies le reste")
else:
    print("❌ avec 0 copie on doit obtenir 0 ; avec 2N copies et w = 1 partout, 2N")

moyenne_tirages = sum(tirages) / len(tirages)
if 1.0 < moyenne_tirages < 1.28:
    print(f"✅ pour 1 copie, {moyenne_tirages:.2f} copie en moyenne à la génération suivante :"
          " l'hétérozygote AS laisse 1 / 0,88 = 1,14 fois plus de descendants que AA")
else:
    print(f"❌ pour 1 copie, on attend environ 1,14 copie en moyenne ; obtenu {moyenne_tirages:.2f}")

## 7. Une mutation, génération après génération

Écrivez `trajectoire(...)`, qui part d'**une copie** et applique `une_generation()` jusqu'à ce que
l'allèle soit perdu, que sa fréquence atteigne un seuil, ou qu'un nombre maximal de générations
soit atteint. Elle renvoie la liste des fréquences.

`seuil=0.10` dans la définition est un **paramètre par défaut** : `trajectoire(N, W_AA, W_AS, W_SS)`
utilise 0,10, `trajectoire(..., seuil=1)` le remplace.

`random.seed(1)` fixe le point de départ du générateur de nombres aléatoires : la même suite de
tirages, donc la même figure, à chaque exécution.

In [ ]:
def trajectoire(N, w_AA, w_AS, w_SS, seuil=0.10, max_generations=300):
    """Suit une mutation neuve (1 copie) génération après génération.
    Renvoie la liste des fréquences, génération 0 comprise."""
    # 1. copies = 1 ; frequences = [copies / (2 * N)]
    # 2. tant que copies > 0, que la fréquence est sous le seuil, et que
    #    len(frequences) <= max_generations :
    #      copies = une_generation(...) ; ajouter la nouvelle fréquence à la liste
    # 3. renvoyer la liste
    pass  # ← remplacez cette ligne par votre code


# 4. random.seed(1), puis tracer sur une même figure 20 trajectoires
#    trajectoire(N, W_AA, W_AS, W_SS, seuil=1, max_generations=150)   (plt.plot)
# 5. titres des axes, titre, plt.show()

In [ ]:
# ✔️ Vérification — exécutez sans modifier
random.seed(3)
essais = [trajectoire(1000, 0.88, 1, 0.14) for _ in range(50)]
if all(t[0] == 1 / 2000 for t in essais):
    print("✅ chaque trajectoire part d'une copie sur 2N : 1 / 2000 = 0,0005")
else:
    print("❌ la liste doit commencer par la fréquence d'une copie, 1 / (2N)")

if all(t[-1] == 0 or t[-1] >= 0.10 or len(t) == 301 for t in essais):
    print("✅ chaque trajectoire s'arrête à la perte, au seuil de 10 %, ou après 300 générations")
else:
    print("❌ une trajectoire doit s'arrêter quand l'allèle est perdu, au seuil, ou après max_generations")

courte = trajectoire(1000, 1, 1, 1, seuil=1, max_generations=5)
if len(courte) <= 6 and (len(courte) == 6 or courte[-1] == 0):
    print("✅ max_generations=5 donne au plus 6 fréquences : générations 0 à 5")
else:
    print("❌ avec max_generations=5, la liste doit contenir au plus 6 fréquences ; obtenu", len(courte))

La plupart des courbes retombent à 0 en quelques générations. Celles qui décollent se stabilisent
vers 0,12 : la fréquence à laquelle l'avantage des AS compense la perte des SS, soit
(1 − w_AA) / ((1 − w_AA) + (1 − w_SS)) = 0,12 / 0,98 ≈ 0,122, proche de la fréquence mesurée à
Ibadan (0,123).

## 8. Mille mutations

Une seule trajectoire ne dit rien : il faut compter. Lancez 1 000 trajectoires **avec paludisme**
(valeurs d'Ibadan) et 1 000 **sans paludisme** (AA et AS ont alors la même valeur sélective :
w_AA = 1). Pour chaque condition : quelle fraction des mutations est perdue ? Pour celles qui
atteignent 10 %, combien de générations faut-il ?

In [ ]:
def repeter(nombre, N, w_AA, w_AS, w_SS):
    """Lance `nombre` trajectoires. Renvoie la fraction de mutations perdues, et la liste
    des nombres de générations des mutations qui ont atteint 10 %."""
    # 1. perdues = 0 ; generations_jusqu_au_seuil = []
    # 2. `nombre` fois : f = trajectoire(N, w_AA, w_AS, w_SS)
    #      si f[-1] == 0 : une mutation perdue de plus
    #      sinon, si f[-1] >= 0.10 : ajouter len(f) - 1 à la liste
    # 3. renvoyer perdues / nombre, et la liste
    pass  # ← remplacez cette ligne par votre code


# 4. random.seed(2)
# 5. perdues_paludisme, generations_paludisme = repeter(1000, N, W_AA, W_AS, W_SS)
# 6. perdues_sans, generations_sans = la même chose sans paludisme (w_AA = 1)
# 7. moyenne_generations_paludisme = moyenne de generations_paludisme
# 8. afficher les deux fractions perdues, le nombre de mutations qui atteignent 10 %,
#    et la moyenne des générations, convertie en années (× 28)

In [ ]:
# ✔️ Vérification — exécutez sans modifier
if 0.70 <= perdues_paludisme <= 0.87:
    print(f"✅ avec paludisme, {perdues_paludisme:.0%} des mutations neuves sont perdues malgré"
          " l'avantage de l'hétérozygote (74,6 % dans les simulations de Shriner et Rotimi 2018)")
else:
    print(f"❌ avec paludisme, on attend 70 à 87 % de mutations perdues ; obtenu {perdues_paludisme:.0%}")

if perdues_sans >= 0.98:
    print(f"✅ sans paludisme, {perdues_sans:.1%} des mutations sont perdues")
else:
    print(f"❌ sans paludisme, on attend au moins 98 % de mutations perdues ; obtenu {perdues_sans:.1%}")

if 30 <= moyenne_generations_paludisme <= 75:
    print(f"✅ {moyenne_generations_paludisme:.0f} générations en moyenne pour atteindre 10 %")
else:
    print(f"❌ on attend 30 à 75 générations en moyenne pour atteindre 10 % ; obtenu"
          f" {moyenne_generations_paludisme:.0f}")

**Question.** Avec paludisme, les AS laissent 14 % de descendants de plus que les AA. Pourquoi la
plupart des mutations neuves sont-elles perdues quand même ?

✍️ *Votre réponse (double-cliquez sur cette cellule pour écrire) :*

**Ce que disent les données réelles.** Shriner et Rotimi (2018) ont fait la même simulation avec
la taille efficace estimée de la population ancestrale, Ne ≈ 25 500 personnes (la **taille
efficace** est l'effectif d'une population idéale, tirée au hasard à chaque génération comme ici,
qui subirait la même dérive). La mutation y est perdue dans **74,6 %** des simulations, et
l'équilibre est atteint en **87 générations** (IC 95 % : 68–124), soit environ 2 400 ans : bien
moins que les 259 générations écoulées depuis l'apparition de la mutation.

Notre population de 1 000 personnes est 25 fois plus petite, pour que la simulation tourne en
quelques secondes. La fraction de mutations perdues dépend peu de N ; le temps d'installation
augmente avec N.

## Bilan

| Question | Résultat |
|---|---|
| Où l'allèle HbS est-il le plus fréquent ? | en Afrique subsaharienne : Gabon 0,280, Nigeria 0,171 |
| Où naissent le plus d'enfants SS ? | Nigeria, Inde, République démocratique du Congo ; région Afrique : 77 % |
| Quelle maladie suit sa répartition ? | le paludisme : r = +0,67 (+0,54 en Afrique) ; tuberculose +0,08, VIH +0,13 ; Piel et al. (2010) trouvent la même relation avec le paludisme de 1900 en Afrique, et aucune dans les Amériques |
| Les porteurs AS sont-ils protégés ? | contre le paludisme grave : rapport des cotes 0,08 (Gambie), 0,09 dans la méta-analyse ; pas contre les autres maladies graves (0,83) |
| Une mutation neuve et avantageuse s'installe-t-elle ? | le plus souvent non : environ 3 sur 4 sont perdues par dérive |
| Et sans paludisme ? | elle est perdue presque à coup sûr |

**Pourquoi cette mutation ?** Là où le paludisme était intense, les AS survivaient mieux que les
AA ; les SS mouraient jeunes. Ces deux effets opposés maintiennent l'allèle à une fréquence
intermédiaire, autour de 10 à 20 %. La mutation est apparue une seule fois, a échappé à la perte
par dérive, puis s'est répandue avec les populations qui la portaient. Là où le paludisme a
disparu, l'avantage des AS disparaît aussi ; la drépanocytose reste.

### Sources

- Piel FB et al. Global epidemiology of sickle haemoglobin in neonates: a contemporary
  geostatistical model-based map and population estimates. *Lancet* 2013;381:142–151.
  doi:10.1016/S0140-6736(12)61229-X — appendix, Web Table 1.
- OMS, Global Health Observatory, année 2000, consulté en septembre 2026 : *Estimated malaria
  incidence (per 1000 population at risk)* (`MALARIA_EST_INCIDENCE`) ; *Incidence of tuberculosis
  (per 100 000 population per year)* (`MDG_0000000020`) ; *Prevalence of HIV among adults aged 15
  to 49 (%)* (`MDG_0000000029`).
- Hill AVS, Allsopp CEM, Kwiatkowski D et al. Common West African HLA antigens are associated
  with protection from severe malaria. *Nature* 1991;352:595–600. doi:10.1038/352595a0 — effectifs
  repris du tableau des études cas-témoins de Taylor et al. 2012.
- Piel FB et al. Global distribution of the sickle cell gene and geographical confirmation of the
  malaria hypothesis. *Nat Commun* 2010;1:104. doi:10.1038/ncomms1104
- Hay SI et al. The global distribution and population at risk of malaria: past, present, and
  future. *Lancet Infect Dis* 2004;4:327–336. doi:10.1016/S1473-3099(04)01043-6
- Taylor SM, Parobek CM, Fairhurst RM. Haemoglobinopathies and the clinical epidemiology of
  malaria. *Lancet Infect Dis* 2012;12:457–468. doi:10.1016/S1473-3099(12)70055-5
- Grosse SD et al. Sickle cell disease in Africa: a neglected cause of early childhood mortality.
  *Am J Prev Med* 2011;41:S398–S405. doi:10.1016/j.amepre.2011.09.013
- Shriner D, Rotimi CN. Whole-genome-sequence-based haplotypes reveal single origin of the sickle
  allele during the Holocene Wet Phase. *Am J Hum Genet* 2018;102:547–556.
  doi:10.1016/j.ajhg.2018.02.003
- Ojodu J et al. Incidence of sickle cell trait — United States, 2010. *MMWR* 2014;63:1155–1158.
- Flint J et al. High frequencies of α-thalassaemia are the result of natural selection by
  malaria. *Nature* 1986;321:744–750. doi:10.1038/321744a0
- Ayala FJ. Evolution — Overdominance. *Encyclopædia Britannica* (effectifs d'Ibadan).